# Deep Q-Networks: Experience Replay and Target Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/deep_q_network_cartpole.ipynb)

**Blog post:** [sesen.ai/blog/deep-q-networks-experience-replay-target-networks](https://sesen.ai/blog/deep-q-networks-experience-replay-target-networks)

In this notebook, we implement a **Deep Q-Network (DQN)** from scratch in PyTorch to solve CartPole-v1. We'll see why two tricks — **experience replay** and a **target network** — are essential for stable training.

**Reference:** Mnih et al., [*Playing Atari with Deep Reinforcement Learning*](https://arxiv.org/abs/1312.5602) (2013) and [*Human-level control through deep reinforcement learning*](https://www.nature.com/articles/nature14236) (2015).

In [ ]:
# Install dependencies (Colab)
# !pip install gymnasium torch matplotlib numpy --quiet

In [ ]:
import numpy as np
import random
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt

# Reproducibility
SEED = 2
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"Gymnasium: {gym.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. The Problem: CartPole

CartPole-v1 is a classic control problem: balance a pole on a moving cart by pushing left or right.

- **State:** 4 continuous values (cart position, cart velocity, pole angle, pole angular velocity)
- **Actions:** Push left (0) or push right (1)
- **Reward:** +1 per timestep the pole stays upright
- **Max score:** 500 (episode truncated)

Unlike FrozenLake's 16 discrete states, CartPole's state space is **continuous** — a Q-table won't work.

In [ ]:
env = gym.make("CartPole-v1")
print(f"State space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"\nSample state: {env.reset(seed=SEED)[0]}")
env.close()

## 2. Building Blocks

### Q-Network

A neural network that takes a state as input and outputs Q-values for each action. Two hidden layers with ReLU activation.

In [ ]:
class QNetwork(nn.Module):
    """Neural network mapping states to Q-values for each action."""

    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, x):
        return self.net(x)


# Quick test
net = QNetwork(state_dim=4, action_dim=2)
dummy_state = torch.randn(1, 4)
q_vals = net(dummy_state)
print(f"Input state:  {dummy_state.numpy().flatten()}")
print(f"Output Q-vals: {q_vals.detach().numpy().flatten()} (left, right)")
print(f"Best action:   {'left' if q_vals.argmax().item() == 0 else 'right'}")

### Experience Replay Buffer

Stores transitions `(s, a, r, s', done)` and samples random minibatches. This breaks the temporal correlation between consecutive experiences.

In [ ]:
class ReplayBuffer:
    """Fixed-size buffer to store experience tuples."""

    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states),
            np.array(actions),
            np.array(rewards, dtype=np.float32),
            np.array(next_states),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)

## 3. The DQN Agent

The agent combines:
1. **Online Q-network** — updated every training step
2. **Target Q-network** — frozen copy, updated every N episodes
3. **Replay buffer** — stores experiences for random sampling; training only starts after the buffer is sufficiently full
4. **Epsilon-greedy** exploration — epsilon stays at 1.0 during buffer collection, then decays linearly during training

The hyperparameters below are adapted from a [working CartPole implementation](https://github.com/zhubarb/sesen_ai_ml_tutorials) that we're building on. In particular:
- **`min_buffer=1500`** — fill the buffer with pure random exploration before any training begins
- **Linear epsilon decay** during training — matching the original's `epsilon -= 1/(n_episodes/0.05)` per training step
- **`gamma=0.95`** — the original author found 0.75 too myopic but 0.95 works well for CartPole's episode lengths

In [ ]:
class DQNAgent:
    def __init__(self, state_dim, action_dim, hidden_dim=128, lr=1e-3,
                 gamma=0.99, buffer_size=10000, batch_size=64,
                 target_update_freq=5, epsilon_start=1.0,
                 epsilon_end=0.05, epsilon_step=0.0005, min_buffer=500):
        self.action_dim = action_dim
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.min_buffer = min_buffer

        # Linear epsilon decay from the original CartPole code:
        # epsilon -= 1.0 / (n_episodes / 0.05) per training step
        # With n_episodes=100: step = 0.0005, decays 1.0 → 0.05 over ~1900 steps
        # Epsilon stays at 1.0 while buffer fills (pure random exploration)
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_step = epsilon_step

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Online network (updated every step)
        self.q_network = QNetwork(state_dim, action_dim, hidden_dim).to(self.device)
        # Target network (frozen copy, updated periodically)
        self.target_network = QNetwork(state_dim, action_dim, hidden_dim).to(self.device)
        self.target_network.load_state_dict(self.q_network.state_dict())

        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_size)

    def select_action(self, state):
        """Epsilon-greedy action selection."""
        if random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
            return self.q_network(state_t).argmax(dim=1).item()

    def train_step(self):
        """Sample a minibatch and perform one gradient step.

        Returns None if the buffer isn't full enough yet (still collecting).
        Epsilon only decays when we actually train — not during collection.
        """
        if len(self.buffer) < max(self.batch_size, self.min_buffer):
            return None

        states, actions, rewards, next_states, dones = self.buffer.sample(
            self.batch_size
        )

        states_t = torch.FloatTensor(states).to(self.device)
        actions_t = torch.LongTensor(actions).to(self.device)
        rewards_t = torch.FloatTensor(rewards).to(self.device)
        next_states_t = torch.FloatTensor(next_states).to(self.device)
        dones_t = torch.FloatTensor(dones).to(self.device)

        # Q(s, a) from online network
        q_values = self.q_network(states_t).gather(
            1, actions_t.unsqueeze(1)
        ).squeeze(1)

        # Target: r + gamma * max_a' Q_target(s', a')
        with torch.no_grad():
            next_q_values = self.target_network(next_states_t).max(dim=1).values
            targets = rewards_t + self.gamma * next_q_values * (1 - dones_t)

        loss = nn.MSELoss()(q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_network.parameters(), 1.0)
        self.optimizer.step()

        # Linear epsilon decay — only during training
        self.epsilon = max(self.epsilon_end, self.epsilon - self.epsilon_step)

        return loss.item()

    def update_target_network(self):
        """Copy online network weights to target network."""
        self.target_network.load_state_dict(self.q_network.state_dict())

## 4. Training (with Early Stopping)

The training loop:
1. Interact with the environment using epsilon-greedy
2. Store every transition in the replay buffer
3. Sample random minibatches and train the online network
4. Periodically copy weights to the target network
5. **Track the best weights and stop early** once the rolling average reward reaches the target

**Why early stopping?** DQN can suffer from *catastrophic forgetting* — the agent reaches near-optimal performance, then continued training destabilises the network and performance collapses. By monitoring the rolling average and saving the best weights, we keep the agent at its peak.

In [ ]:
import copy

def train_dqn(n_episodes=500, seed=SEED, early_stop_reward=400,
              early_stop_window=50, **agent_kwargs):
    """Train a DQN agent with early stopping.

    Stops training when the rolling average reward over `early_stop_window`
    episodes reaches `early_stop_reward`. Restores the best weights seen
    during training to guard against catastrophic forgetting.

    Set early_stop_reward=None to disable early stopping.
    """
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)

    env = gym.make("CartPole-v1")
    agent = DQNAgent(
        state_dim=env.observation_space.shape[0],
        action_dim=env.action_space.n,
        **agent_kwargs,
    )

    rewards_history = []
    best_avg_reward = -float('inf')
    best_weights = None

    for episode in range(n_episodes):
        state, _ = env.reset(seed=seed + episode)
        total_reward = 0

        for step in range(500):
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.buffer.push(state, action, reward, next_state, done)
            agent.train_step()

            state = next_state
            total_reward += reward
            if done:
                break

        if episode % agent.target_update_freq == 0:
            agent.update_target_network()

        rewards_history.append(total_reward)

        # Track best weights
        if len(rewards_history) >= early_stop_window:
            avg = np.mean(rewards_history[-early_stop_window:])
            if avg > best_avg_reward:
                best_avg_reward = avg
                best_weights = copy.deepcopy(agent.q_network.state_dict())

            # Early stopping
            if early_stop_reward is not None and avg >= early_stop_reward:
                print(f"*** Early stopping at episode {episode} "
                      f"(avg reward: {avg:.1f}) ***")
                break

        if episode % 50 == 0:
            avg = (np.mean(rewards_history[-50:]) if len(rewards_history) >= 50
                   else np.mean(rewards_history))
            print(f"Episode {episode:3d} | Avg reward: {avg:6.1f} | "
                  f"Epsilon: {agent.epsilon:.3f} | Buffer: {len(agent.buffer)}")

    # Restore best weights to guard against late-training instability
    if best_weights is not None:
        agent.q_network.load_state_dict(best_weights)
        print(f"Restored best weights (avg reward: {best_avg_reward:.1f})")

    env.close()
    return rewards_history, agent


# Train!
rewards_full, agent_full = train_dqn(n_episodes=500, early_stop_reward=400)
print(f"\nFinal avg reward (last 50): {np.mean(rewards_full[-50:]):.1f}")

## 5. Visualise the Learning

In [ ]:
rolling = np.convolve(rewards_full, np.ones(20)/20, mode='valid')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rolling, '#2196F3', linewidth=1.0)
ax.axhline(y=500, color='#4CAF50', linestyle='--', alpha=0.5, label='Max score (500)')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (20-episode rolling avg)')
ax.set_title('DQN on CartPole-v1')
ax.set_ylim(0, 550)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 6. Ablation: Why Experience Replay Matters

Let's see what happens when we remove experience replay — using a tiny 64-transition buffer that can't break temporal correlations.

In [ ]:
print("Training: No experience replay (tiny buffer)...")
rewards_no_replay, _ = train_dqn(
    n_episodes=500, early_stop_reward=None,
    buffer_size=64, batch_size=8, min_buffer=8
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
window = 20

r_full = np.convolve(rewards_full, np.ones(window)/window, mode='valid')
r_no_replay = np.convolve(rewards_no_replay, np.ones(window)/window, mode='valid')

ax.plot(r_full, '#2196F3', linewidth=1.5, label='Full DQN (early stopped)')
ax.plot(r_no_replay, '#F44336', linewidth=1.5, linestyle=':', label='No experience replay')
ax.axhline(y=500, color='#4CAF50', linestyle='--', alpha=0.3, label='Max score (500)')
ax.set_xlabel('Episode')
ax.set_ylabel(f'Reward ({window}-ep rolling avg)')
ax.set_title('DQN Ablation: Experience Replay')
ax.set_ylim(0, 550)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"Full DQN (early stopped) final avg: {np.mean(rewards_full[-50:]):.1f}")
print(f"No replay final avg:                {np.mean(rewards_no_replay[-50:]):.1f}")

## 7. Hyperparameter Sensitivity

### Discount factor (gamma)

In [ ]:
gammas = [0.5, 0.9, 0.99, 0.999]
gamma_results = {}

for g in gammas:
    print(f"Training with gamma={g}...")
    rewards, _ = train_dqn(n_episodes=300, early_stop_reward=None, gamma=g)
    gamma_results[g] = rewards

fig, ax = plt.subplots(figsize=(8, 4))
for g, rewards in gamma_results.items():
    r = np.convolve(rewards, np.ones(20)/20, mode='valid')
    ax.plot(r, label=f'gamma={g}', linewidth=1.2)

ax.set_xlabel('Episode')
ax.set_ylabel('Reward (20-ep rolling avg)')
ax.set_title('Effect of Discount Factor (gamma)')
ax.set_ylim(0, 550)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

### Epsilon decay speed

In [ ]:
# Visualise the epsilon schedule used in our agent
# Linear decay: epsilon -= (start - end) / (n_episodes / 0.05)  per training step
# With n_episodes=300: step_size = 0.95/6000 ≈ 0.000158
# Training starts after min_buffer=1500 transitions are collected

fig, ax = plt.subplots(figsize=(8, 3))

for n_ep, label in [(100, 'n_episodes=100 (original)'),
                     (300, 'n_episodes=300 (ours)'),
                     (500, 'n_episodes=500')]:
    step_size = 0.95 / (n_ep / 0.05)
    n_steps = int(0.95 / step_size) + 500  # a bit past convergence
    steps = np.arange(n_steps)
    eps = np.maximum(0.05, 1.0 - steps * step_size)
    ax.plot(steps, eps, label=label, linewidth=1.2)

ax.set_xlabel('Training steps (after buffer fills)')
ax.set_ylabel('Epsilon')
ax.set_title('Linear Epsilon Decay: epsilon -= 1/(n_episodes/0.05)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 8. Exercises

1. **Gamma sweep** — Try `gamma` values of 0.5, 0.9, 0.99, and 0.999. How does the discount factor affect convergence speed and final performance?

2. **Network size** — Try `hidden_dim` of 32, 64, 128, and 256. Is bigger always better for CartPole?

3. **Soft target updates (Polyak averaging)** — Instead of copying weights every N episodes, blend them every step:
   ```python
   tau = 0.005
   for target_param, param in zip(target_net.parameters(), online_net.parameters()):
       target_param.data.copy_(tau * param.data + (1.0 - tau) * target_param.data)
   ```
   Does this improve stability?

4. **Double DQN** — Modify the target computation to use the online network for action selection:
   ```python
   # Instead of: next_q = target_net(s').max()
   best_actions = online_net(next_states).argmax(dim=1)
   next_q = target_net(next_states).gather(1, best_actions.unsqueeze(1)).squeeze()
   ```
   Does this reduce Q-value overestimation?

5. **Prioritised replay** — Instead of uniform sampling, sample transitions proportional to their TD error. Does this speed up learning?

## References

- Mnih et al., [*Playing Atari with Deep Reinforcement Learning*](https://arxiv.org/abs/1312.5602) (2013)
- Mnih et al., [*Human-level control through deep reinforcement learning*](https://www.nature.com/articles/nature14236) (2015)
- Sutton & Barto, [*Reinforcement Learning: An Introduction*](http://incompleteideas.net/book/the-book-2nd.html) (2018)
- Lin, [*Self-Improving Reactive Agents Based on Reinforcement Learning*](https://link.springer.com/article/10.1007/BF00992699) (1992)